# Load packages and libraries

In [1]:
.libPaths()
assign(".lib.loc", "/home/manuel.tardaguila/conda_envs/multiome_NEW_downstream_analysis/lib/R/library", envir = environment(.libPaths))
.libPaths()
# sessionInfo()

Sys.setenv(RETICULATE_PYTHON="/home/manuel.tardaguila/conda_envs/multiome_NEW_downstream_analysis/bin/python")
library(reticulate)
reticulate::use_python("/home/manuel.tardaguila/conda_envs/multiome_NEW_downstream_analysis/bin/python")
reticulate::use_condaenv("/home/manuel.tardaguila/conda_envs/multiome_NEW_downstream_analysis")
reticulate::py_module_available(module='leidenalg')
reticulate::import('leidenalg') 
suppressMessages(library(hdf5r))
suppressMessages(library(Seurat)) 
suppressMessages(library(Signac)) 
suppressMessages(library(EnsDb.Hsapiens.v86)) 
suppressMessages(library(dplyr)) 
suppressMessages(library(ggplot2)) 
suppressMessages(library(Matrix)) 
suppressMessages(library(data.table)) 
suppressMessages(library(ggpubr)) 
suppressMessages(library(ggplot2))
suppressMessages(library(chromVAR))
suppressMessages(library(enrichR))
suppressMessages(library(JASPAR2020))
suppressMessages(library(TFBSTools))
suppressMessages(library(motifmatchr))
suppressMessages(library(BSgenome.Hsapiens.UCSC.hg38))
library(pheatmap)

[1] "/group/soranzo/conda_envs/multiome_NEW_downstream_analysis/lib/R/library"

[1] "/home/manuel.tardaguila/conda_envs/multiome_NEW_downstream_analysis/lib/R/library"

[1] TRUE

Module(leidenalg)

# Read in data

In [2]:
setwd("/group/soranzo/manuel.tardaguila/2025_K562_multiome_reanalysis/Downstream_analysis/")

In [3]:
adata<-readRDS(file="merged_clusters_final_annotated.rds")

In [4]:
adata

An object of class Seurat 
459097 features across 11250 samples within 4 assays 
Active assay: SCT (29123 features, 3000 variable features)
 3 layers present: counts, data, scale.data
 3 other assays present: RNA, RNA_raw, ATAC
 5 dimensional reductions calculated: pca, umap.rna, lsi, umap.atac, umap.wnn

# Find motifs in peaks

In [5]:
DefaultAssay(adata) <- 'ATAC'

In [6]:
pfm <- getMatrixSet(
  x = JASPAR2020,
  opts = list(collection = "CORE", tax_group = 'vertebrates', all_versions = FALSE))

In [7]:
#str(pfm)

In [8]:
motifdata = cbind(sapply(pfm, function(x) unlist(x@name)),sapply(pfm, function(x) unlist(x@matrixClass )))

In [9]:
#str(motifdata)

In [10]:
TF_motifs<-unlist(motifdata)

TF_motifs[grep("CUX1|RUNX1",TF_motifs)]

[1] "RUNX1" "CUX1"

In [11]:
names(pfm) <- motifdata[,1]

# Add to adata

In [12]:
adata <- AddMotifs(
  object = adata, genome =BSgenome.Hsapiens.UCSC.hg38 , assay= "ATAC",
  pfm = pfm
)

Building motif matrix

Finding motif positions

Creating Motif object



In [13]:
#str(adata)

# chromVar step

In [14]:
#### long step
DefaultAssay(adata) <- 'ATAC'
adata <- RunChromVAR(
  object = adata,
  genome = BSgenome.Hsapiens.UCSC.hg38
)

Computing GC bias per region

Selecting background regions

Computing deviations from background

Constructing chromVAR assay

Warning message:
"Layer counts isn't present in the assay object; returning NULL"


In [15]:
DefaultAssay(adata) <- 'chromvar'

In [16]:
devscores = GetAssayData(adata,layer='data', assay="chromvar")

In [17]:
str(devscores)

 num [1:746, 1:11250] -1.1901 -0.7441 0.0309 -1.2753 -3.6719 ...
 - attr(*, "dimnames")=List of 2
  ..$ : chr [1:746] "Arnt" "Ahr::Arnt" "Ddit3::Cebpa" "Mecom" ...
  ..$ : chr [1:11250] "MCO_1278_AAACAGCCAAGGTCCT-1" "MCO_1278_AAACAGCCATGGTTAT-1" "MCO_1278_AAACATGCAGAAATGC-1" "MCO_1278_AAACCAACACATAACT-1" ...


In [18]:
which(row.names(devscores) == 'CUX1')

which(row.names(devscores) == 'RUNX1')

[1] 243

[1] 41

In [19]:
#devscores[243,]

# Save data

In [20]:
output_dir<-"/group/soranzo/manuel.tardaguila/2025_K562_multiome_reanalysis/Downstream_analysis/chromvar_analysis/"

In [21]:
write.table(devscores, file=file.path(output_dir,'chromVAR_dev_scores_j2020.txt'), sep='\t', quote=FALSE)


In [35]:
setwd("/group/soranzo/manuel.tardaguila/2025_K562_multiome_reanalysis/Downstream_analysis/")

In [36]:
saveRDS(adata, file="merged_clusters_final_annotated_motifs_and_chromvar.rds")

# Pick-up after calculation of scores

In [14]:
setwd("/group/soranzo/manuel.tardaguila/2025_K562_multiome_reanalysis/Downstream_analysis/")

adata<-readRDS(file="merged_clusters_final_annotated_motifs_and_chromvar.rds")

adata

An object of class Seurat 
459843 features across 11250 samples within 5 assays 
Active assay: chromvar (746 features, 0 variable features)
 1 layer present: data
 4 other assays present: RNA, RNA_raw, ATAC, SCT
 5 dimensional reductions calculated: pca, umap.rna, lsi, umap.atac, umap.wnn

# chromvar analysis

## Subset to cluster 1

In [15]:
adata_sub_cluster_1<-subset(adata, seurat_clusters == '1')

adata_sub_cluster_1

Idents(adata_sub_cluster_1)<- "Genotype"


summary(adata_sub_cluster_1@meta.data$Genotype)


An object of class Seurat 
459843 features across 2811 samples within 5 assays 
Active assay: chromvar (746 features, 0 variable features)
 1 layer present: data
 4 other assays present: RNA, RNA_raw, ATAC, SCT
 5 dimensional reductions calculated: pca, umap.rna, lsi, umap.atac, umap.wnn

wt rs139141690_HET     rs139141690        Del_16bp        Del_80bp 
            834             494             396             172             915

In [16]:
results_cluster_1<-data.frame()

### wt vs Del_16bp

In [17]:
# Set the chromvar assay as the default for this analysis
DefaultAssay(adata_sub_cluster_1) <- "chromvar"

# Find differentially active motifs between two cell types (e.g., "T cells" vs "B cells")
# This is analogous to FindMarkers for gene expression, but now on motif deviation scores
diff_motifs <- FindMarkers(
  object = adata_sub_cluster_1,
  ident.1 = "wt",
  ident.2 = "Del_16bp",
  group.by = "Genotype", # Or whatever metadata column defines your groups
  test.use = "t",    # Or "bimod", "t", etc.
  min.pct = 0.1,          # Minimum percentage of cells in either group expressing the feature
  logfc.threshold = 0.25  # Minimum log2 fold-change for motif deviation
)

Warning message:
"The `slot` argument of `GetAssayData()` is deprecated as of SeuratObject 5.0.0.
ℹ Please use the `layer` argument instead.
ℹ The deprecated feature was likely used in the Seurat package.
  Please report the issue at <https://github.com/satijalab/seurat/issues>."


In [18]:
diff_motifs$motif<-row.names(diff_motifs)

row.names(diff_motifs)<-NULL

In [19]:

diff_motifs$comparison<-'Genotype_Del_16bp_vs_wt'
diff_motifs$identity<-'1'

str(diff_motifs)
cat("\n")

results_cluster_1<-rbind(diff_motifs,results_cluster_1)


'data.frame':	400 obs. of  8 variables:
 $ p_val     : num  4.36e-08 4.48e-08 1.64e-07 6.91e-07 8.61e-07 ...
 $ avg_log2FC: num  10.42 6.36 6.1 5.39 6.16 ...
 $ pct.1     : num  0.721 0.673 0.665 0.67 0.629 0.637 0.297 0.428 0.348 0.451 ...
 $ pct.2     : num  0.506 0.471 0.477 0.494 0.442 0.436 0.5 0.645 0.541 0.657 ...
 $ p_val_adj : num  3.25e-05 3.34e-05 1.22e-04 5.15e-04 6.43e-04 ...
 $ motif     : chr  "GATA1::TAL1" "GATA4" "GATA2" "GATA5" ...
 $ comparison: chr  "Genotype_Del_16bp_vs_wt" "Genotype_Del_16bp_vs_wt" "Genotype_Del_16bp_vs_wt" "Genotype_Del_16bp_vs_wt" ...
 $ identity  : chr  "1" "1" "1" "1" ...



In [20]:
indexes<-grep("CUX1|RUNX1",diff_motifs$motif)
diff_motifs[indexes,]

,p_val,avg_log2FC,pct.1,pct.2,p_val_adj,motif,comparison,identity
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>
92,0.02065796,6.061064,0.366,0.453,1,RUNX1,Genotype_Del_16bp_vs_wt,1


### wt vs Del_80bp

In [21]:
# Set the chromvar assay as the default for this analysis
DefaultAssay(adata_sub_cluster_1) <- "chromvar"

# Find differentially active motifs between two cell types (e.g., "T cells" vs "B cells")
# This is analogous to FindMarkers for gene expression, but now on motif deviation scores
diff_motifs <- FindMarkers(
  object = adata_sub_cluster_1,
  ident.1 = "wt",
  ident.2 = "Del_80bp",
  group.by = "Genotype", # Or whatever metadata column defines your groups
  test.use = "t",    # Or "bimod", "t", etc.
  min.pct = 0.1,          # Minimum percentage of cells in either group expressing the feature
  logfc.threshold = 0.25  # Minimum log2 fold-change for motif deviation
)

In [22]:
diff_motifs$motif<-row.names(diff_motifs)

row.names(diff_motifs)<-NULL

In [23]:

diff_motifs$comparison<-'Genotype_Del_80bp_vs_wt'
diff_motifs$identity<-'1'

str(diff_motifs)
cat("\n")

results_cluster_1<-rbind(diff_motifs,results_cluster_1)


'data.frame':	370 obs. of  8 variables:
 $ p_val     : num  2.23e-23 2.51e-23 4.68e-22 5.84e-22 9.24e-22 ...
 $ avg_log2FC: num  -2.17 -3.36 -2.11 -2.32 -3.22 ...
 $ pct.1     : num  0.288 0.376 0.375 0.297 0.348 0.393 0.394 0.385 0.397 0.387 ...
 $ pct.2     : num  0.525 0.564 0.538 0.499 0.525 0.552 0.553 0.554 0.55 0.55 ...
 $ p_val_adj : num  1.66e-20 1.87e-20 3.49e-19 4.35e-19 6.89e-19 ...
 $ motif     : chr  "GABPA" "ERF" "ETV5" "ELF1" ...
 $ comparison: chr  "Genotype_Del_80bp_vs_wt" "Genotype_Del_80bp_vs_wt" "Genotype_Del_80bp_vs_wt" "Genotype_Del_80bp_vs_wt" ...
 $ identity  : chr  "1" "1" "1" "1" ...



In [24]:
indexes<-grep("CUX1|RUNX1",diff_motifs$motif)
diff_motifs[indexes,]

p_val,avg_log2FC,pct.1,pct.2,p_val_adj,motif,comparison,identity
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>


### wt vs rs139141690

In [25]:
# Set the chromvar assay as the default for this analysis
DefaultAssay(adata_sub_cluster_1) <- "chromvar"

# Find differentially active motifs between two cell types (e.g., "T cells" vs "B cells")
# This is analogous to FindMarkers for gene expression, but now on motif deviation scores
diff_motifs <- FindMarkers(
  object = adata_sub_cluster_1,
  ident.1 = "wt",
  ident.2 = "rs139141690",
  group.by = "Genotype", # Or whatever metadata column defines your groups
  test.use = "t",    # Or "bimod", "t", etc.
  min.pct = 0.1,          # Minimum percentage of cells in either group expressing the feature
  logfc.threshold = 0.25  # Minimum log2 fold-change for motif deviation
)

In [26]:
diff_motifs$motif<-row.names(diff_motifs)

row.names(diff_motifs)<-NULL

In [27]:

diff_motifs$comparison<-'Genotype_rs139141690_vs_wt'
diff_motifs$identity<-'1'

str(diff_motifs)
cat("\n")

results_cluster_1<-rbind(diff_motifs,results_cluster_1)


'data.frame':	345 obs. of  8 variables:
 $ p_val     : num  4.03e-18 2.67e-16 3.01e-16 5.26e-16 4.20e-15 ...
 $ avg_log2FC: num  -3.5 -2.24 -2.27 -3.86 -2.21 ...
 $ pct.1     : num  0.348 0.375 0.412 0.376 0.398 0.394 0.385 0.403 0.302 0.354 ...
 $ pct.2     : num  0.535 0.571 0.614 0.576 0.616 0.578 0.588 0.591 0.503 0.533 ...
 $ p_val_adj : num  3.01e-15 1.99e-13 2.24e-13 3.93e-13 3.13e-12 ...
 $ motif     : chr  "ETV6" "ETV5" "ETS2" "ERF" ...
 $ comparison: chr  "Genotype_rs139141690_vs_wt" "Genotype_rs139141690_vs_wt" "Genotype_rs139141690_vs_wt" "Genotype_rs139141690_vs_wt" ...
 $ identity  : chr  "1" "1" "1" "1" ...



In [28]:
indexes<-grep("CUX1|RUNX1",diff_motifs$motif)
diff_motifs[indexes,]

,p_val,avg_log2FC,pct.1,pct.2,p_val_adj,motif,comparison,identity
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>
61,1.019454e-08,-3.2781310,0.366,0.530,7.605128e-06,RUNX1,Genotype_rs139141690_vs_wt,1
122,1.914905e-04,-0.2998634,0.537,0.619,1.428519e-01,CUX1,Genotype_rs139141690_vs_wt,1


### Del_80bp vs rs139141690

In [243]:
# Set the chromvar assay as the default for this analysis
DefaultAssay(adata_sub_cluster_1) <- "chromvar"

# Find differentially active motifs between two cell types (e.g., "T cells" vs "B cells")
# This is analogous to FindMarkers for gene expression, but now on motif deviation scores
diff_motifs <- FindMarkers(
  object = adata_sub_cluster_1,
  ident.1 = "Del_80bp",
  ident.2 = "rs139141690",
  group.by = "Genotype", # Or whatever metadata column defines your groups
  test.use = "t",    # Or "bimod", "t", etc.
  min.pct = 0.1,          # Minimum percentage of cells in either group expressing the feature
  logfc.threshold = 0.25  # Minimum log2 fold-change for motif deviation
)

In [244]:
diff_motifs$motif<-row.names(diff_motifs)

row.names(diff_motifs)<-NULL

In [245]:

diff_motifs$comparison<-'Genotype_rs139141690_vs_Del_80bp'
diff_motifs$identity<-'1'

str(diff_motifs)
cat("\n")

results_cluster_1<-rbind(diff_motifs,results_cluster_1)


'data.frame':	331 obs. of  8 variables:
 $ p_val     : num  4.95e-25 4.56e-22 9.21e-22 1.10e-20 1.27e-20 ...
 $ avg_log2FC: num  7.944 9.564 -0.747 7.391 6.934 ...
 $ pct.1     : num  0.499 0.522 0.485 0.561 0.545 0.555 0.51 0.475 0.38 0.402 ...
 $ pct.2     : num  0.768 0.773 0.712 0.77 0.78 0.768 0.659 0.621 0.298 0.548 ...
 $ p_val_adj : num  3.69e-22 3.40e-19 6.87e-19 8.22e-18 9.49e-18 ...
 $ motif     : chr  "GATA6" "GATA3" "GATA1" "GATA4" ...
 $ comparison: chr  "Genotype_rs139141690_vs_Del_80bp" "Genotype_rs139141690_vs_Del_80bp" "Genotype_rs139141690_vs_Del_80bp" "Genotype_rs139141690_vs_Del_80bp" ...
 $ identity  : chr  "1" "1" "1" "1" ...



In [246]:
indexes<-grep("CUX1|RUNX1",diff_motifs$motif)
diff_motifs[indexes,]

,p_val,avg_log2FC,pct.1,pct.2,p_val_adj,motif,comparison,identity
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>
32,1.666772e-06,-0.364203,0.523,0.619,0.001243412,CUX1,Genotype_rs139141690_vs_Del_80bp,1
309,7.259852e-01,-3.524199,0.525,0.530,1.000000000,RUNX1,Genotype_rs139141690_vs_Del_80bp,1


### Evaluate

In [248]:
indexes<-grep("CUX1|RUNX1",results_cluster_1$motif)
results_cluster_1[indexes,]



,p_val,avg_log2FC,pct.1,pct.2,p_val_adj,motif,comparison,identity
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>
32,1.666772e-06,-0.3642030,0.523,0.619,1.243412e-03,CUX1,Genotype_rs139141690_vs_Del_80bp,1
309,7.259852e-01,-3.5241991,0.525,0.530,1.000000e+00,RUNX1,Genotype_rs139141690_vs_Del_80bp,1
381,2.655619e-02,5.8149963,0.525,0.453,1.000000e+00,RUNX1,Genotype_Del_16bp_vs_Del_80bp,1
701,1.019454e-08,-3.2781310,0.366,0.530,7.605128e-06,RUNX1,Genotype_rs139141690_vs_wt,1
762,1.914905e-04,-0.2998634,0.537,0.619,1.428519e-01,CUX1,Genotype_rs139141690_vs_wt,1
1447,2.065796e-02,6.0610644,0.366,0.453,1.000000e+00,RUNX1,Genotype_Del_16bp_vs_wt,1


## Subset to cluster 3

In [249]:
adata_sub_cluster_3<-subset(adata, seurat_clusters == '3')

adata_sub_cluster_3

Idents(adata_sub_cluster_3)<- "Genotype"


summary(adata_sub_cluster_3@meta.data$Genotype)


An object of class Seurat 
459843 features across 1584 samples within 5 assays 
Active assay: chromvar (746 features, 0 variable features)
 1 layer present: data
 4 other assays present: RNA, RNA_raw, ATAC, SCT
 5 dimensional reductions calculated: pca, umap.rna, lsi, umap.atac, umap.wnn

wt rs139141690_HET     rs139141690        Del_16bp        Del_80bp 
            454             132             379             111             508

In [250]:
results_cluster_3<-data.frame()

### wt vs Del_16bp

In [251]:
# Set the chromvar assay as the default for this analysis
DefaultAssay(adata_sub_cluster_3) <- "chromvar"

# Find differentially active motifs between two cell types (e.g., "T cells" vs "B cells")
# This is analogous to FindMarkers for gene expression, but now on motif deviation scores
diff_motifs <- FindMarkers(
  object = adata_sub_cluster_3,
  ident.1 = "wt",
  ident.2 = "Del_16bp",
  group.by = "Genotype", # Or whatever metadata column defines your groups
  test.use = "t",    # Or "bimod", "t", etc.
  min.pct = 0.1,          # Minimum percentage of cells in either group expressing the feature
  logfc.threshold = 0.25  # Minimum log2 fold-change for motif deviation
)

In [252]:
diff_motifs$motif<-row.names(diff_motifs)

row.names(diff_motifs)<-NULL

In [253]:

diff_motifs$comparison<-'Genotype_Del_16bp_vs_wt'
diff_motifs$identity<-'3'

str(diff_motifs)
cat("\n")

results_cluster_3<-rbind(diff_motifs,results_cluster_3)


'data.frame':	520 obs. of  8 variables:
 $ p_val     : num  4.21e-09 5.24e-09 8.11e-09 1.82e-08 3.18e-08 ...
 $ avg_log2FC: num  -2.14 -1.84 -1.52 -1.6 -4.3 ...
 $ pct.1     : num  0.381 0.39 0.396 0.385 0.399 0.381 0.39 0.319 0.482 0.361 ...
 $ pct.2     : num  0.649 0.685 0.676 0.667 0.685 0.631 0.649 0.568 0.703 0.586 ...
 $ p_val_adj : num  3.14e-06 3.91e-06 6.05e-06 1.36e-05 2.37e-05 ...
 $ motif     : chr  "ERF" "ELK4" "ETS1" "ZBTB7A" ...
 $ comparison: chr  "Genotype_Del_16bp_vs_wt" "Genotype_Del_16bp_vs_wt" "Genotype_Del_16bp_vs_wt" "Genotype_Del_16bp_vs_wt" ...
 $ identity  : chr  "3" "3" "3" "3" ...



In [254]:
indexes<-grep("CUX1|RUNX1",diff_motifs$motif)
diff_motifs[indexes,]

,p_val,avg_log2FC,pct.1,pct.2,p_val_adj,motif,comparison,identity
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>
14,1.244185e-06,-2.668261,0.588,0.757,0.0009281622,RUNX1,Genotype_Del_16bp_vs_wt,3


### wt vs Del_80bp

In [255]:
# Set the chromvar assay as the default for this analysis
DefaultAssay(adata_sub_cluster_3) <- "chromvar"

# Find differentially active motifs between two cell types (e.g., "T cells" vs "B cells")
# This is analogous to FindMarkers for gene expression, but now on motif deviation scores
diff_motifs <- FindMarkers(
  object = adata_sub_cluster_3,
  ident.1 = "wt",
  ident.2 = "Del_80bp",
  group.by = "Genotype", # Or whatever metadata column defines your groups
  test.use = "t",    # Or "bimod", "t", etc.
  min.pct = 0.1,          # Minimum percentage of cells in either group expressing the feature
  logfc.threshold = 0.25  # Minimum log2 fold-change for motif deviation
)

In [256]:
diff_motifs$motif<-row.names(diff_motifs)

row.names(diff_motifs)<-NULL

In [257]:

diff_motifs$comparison<-'Genotype_Del_80bp_vs_wt'
diff_motifs$identity<-'3'

str(diff_motifs)
cat("\n")

results_cluster_3<-rbind(diff_motifs,results_cluster_3)


'data.frame':	488 obs. of  8 variables:
 $ p_val     : num  3.25e-32 2.30e-31 9.40e-31 1.63e-30 4.71e-30 ...
 $ avg_log2FC: num  -2.37 -2.19 -3.11 -4.24 -1.37 ...
 $ pct.1     : num  0.381 0.399 0.291 0.319 0.39 0.392 0.396 0.381 0.385 0.308 ...
 $ pct.2     : num  0.683 0.687 0.654 0.657 0.689 0.697 0.663 0.679 0.673 0.64 ...
 $ p_val_adj : num  2.43e-29 1.71e-28 7.01e-28 1.22e-27 3.51e-27 ...
 $ motif     : chr  "ERF" "ETV5" "ETV1" "GABPA" ...
 $ comparison: chr  "Genotype_Del_80bp_vs_wt" "Genotype_Del_80bp_vs_wt" "Genotype_Del_80bp_vs_wt" "Genotype_Del_80bp_vs_wt" ...
 $ identity  : chr  "3" "3" "3" "3" ...



In [258]:
indexes<-grep("CUX1|RUNX1",diff_motifs$motif)
diff_motifs[indexes,]

,p_val,avg_log2FC,pct.1,pct.2,p_val_adj,motif,comparison,identity
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>
21,2.184512e-16,-3.0815370,0.588,0.783,1.629646e-13,RUNX1,Genotype_Del_80bp_vs_wt,3
412,7.259871e-01,-0.2560605,0.630,0.640,1.000000e+00,CUX1,Genotype_Del_80bp_vs_wt,3


### wt vs rs139141690

In [259]:
# Set the chromvar assay as the default for this analysis
DefaultAssay(adata_sub_cluster_3) <- "chromvar"

# Find differentially active motifs between two cell types (e.g., "T cells" vs "B cells")
# This is analogous to FindMarkers for gene expression, but now on motif deviation scores
diff_motifs <- FindMarkers(
  object = adata_sub_cluster_3,
  ident.1 = "wt",
  ident.2 = "rs139141690",
  group.by = "Genotype", # Or whatever metadata column defines your groups
  test.use = "t",    # Or "bimod", "t", etc.
  min.pct = 0.1,          # Minimum percentage of cells in either group expressing the feature
  logfc.threshold = 0.25  # Minimum log2 fold-change for motif deviation
)

In [260]:
diff_motifs$motif<-row.names(diff_motifs)

row.names(diff_motifs)<-NULL

In [261]:

diff_motifs$comparison<-'Genotype_rs139141690_vs_wt'
diff_motifs$identity<-'3'

str(diff_motifs)
cat("\n")

results_cluster_3<-rbind(diff_motifs,results_cluster_3)


'data.frame':	361 obs. of  8 variables:
 $ p_val     : num  5.82e-23 7.30e-22 7.92e-21 8.85e-21 8.76e-20 ...
 $ avg_log2FC: num  -1.78 -1.73 -2.73 -4.56 -1.08 ...
 $ pct.1     : num  0.392 0.352 0.319 0.392 0.399 0.291 0.515 0.308 0.454 0.39 ...
 $ pct.2     : num  0.652 0.615 0.586 0.67 0.67 0.562 0.739 0.554 0.675 0.665 ...
 $ p_val_adj : num  4.34e-20 5.44e-19 5.91e-18 6.61e-18 6.53e-17 ...
 $ motif     : chr  "ETV6" "ELF3" "GABPA" "EHF" ...
 $ comparison: chr  "Genotype_rs139141690_vs_wt" "Genotype_rs139141690_vs_wt" "Genotype_rs139141690_vs_wt" "Genotype_rs139141690_vs_wt" ...
 $ identity  : chr  "3" "3" "3" "3" ...



In [262]:
indexes<-grep("CUX1|RUNX1",diff_motifs$motif)
diff_motifs[indexes,]

,p_val,avg_log2FC,pct.1,pct.2,p_val_adj,motif,comparison,identity
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>
31,7.008065e-07,-0.4635739,0.588,0.744,0.0005228016,RUNX1,Genotype_rs139141690_vs_wt,3
151,1.498794e-02,-0.5547957,0.630,0.697,1.0000000000,CUX1,Genotype_rs139141690_vs_wt,3


### Del_80bp vs Del_16bp

In [263]:
# Set the chromvar assay as the default for this analysis
DefaultAssay(adata_sub_cluster_3) <- "chromvar"

# Find differentially active motifs between two cell types (e.g., "T cells" vs "B cells")
# This is analogous to FindMarkers for gene expression, but now on motif deviation scores
diff_motifs <- FindMarkers(
  object = adata_sub_cluster_3,
  ident.1 = "Del_80bp",
  ident.2 = "Del_16bp",
  group.by = "Genotype", # Or whatever metadata column defines your groups
  test.use = "t",    # Or "bimod", "t", etc.
  min.pct = 0.1,          # Minimum percentage of cells in either group expressing the feature
  logfc.threshold = 0.25  # Minimum log2 fold-change for motif deviation
)

In [264]:
diff_motifs$motif<-row.names(diff_motifs)

row.names(diff_motifs)<-NULL

In [265]:

diff_motifs$comparison<-'Genotype_Del_16bp_vs_Del_80bp'
diff_motifs$identity<-'3'

str(diff_motifs)
cat("\n")

results_cluster_3<-rbind(diff_motifs,results_cluster_3)


'data.frame':	443 obs. of  8 variables:
 $ p_val     : num  0.000795 0.001257 0.00573 0.00862 0.010279 ...
 $ avg_log2FC: num  -0.336 -1.062 0.664 0.972 -3.175 ...
 $ pct.1     : num  0.352 0.333 0.557 0.396 0.565 0.411 0.437 0.417 0.585 0.494 ...
 $ pct.2     : num  0.477 0.45 0.459 0.486 0.712 0.523 0.541 0.532 0.667 0.577 ...
 $ p_val_adj : num  0.593 0.938 1 1 1 ...
 $ motif     : chr  "SREBF2" "NR2F1" "OTX1" "ZNF740" ...
 $ comparison: chr  "Genotype_Del_16bp_vs_Del_80bp" "Genotype_Del_16bp_vs_Del_80bp" "Genotype_Del_16bp_vs_Del_80bp" "Genotype_Del_16bp_vs_Del_80bp" ...
 $ identity  : chr  "3" "3" "3" "3" ...



In [266]:
indexes<-grep("CUX1|RUNX1",diff_motifs$motif)
diff_motifs[indexes,]

,p_val,avg_log2FC,pct.1,pct.2,p_val_adj,motif,comparison,identity
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>
222,0.3756347,0.4132764,0.783,0.757,1,RUNX1,Genotype_Del_16bp_vs_Del_80bp,3


### Del_80bp vs rs139141690

In [267]:
# Set the chromvar assay as the default for this analysis
DefaultAssay(adata_sub_cluster_3) <- "chromvar"

# Find differentially active motifs between two cell types (e.g., "T cells" vs "B cells")
# This is analogous to FindMarkers for gene expression, but now on motif deviation scores
diff_motifs <- FindMarkers(
  object = adata_sub_cluster_3,
  ident.1 = "Del_80bp",
  ident.2 = "rs139141690",
  group.by = "Genotype", # Or whatever metadata column defines your groups
  test.use = "t",    # Or "bimod", "t", etc.
  min.pct = 0.1,          # Minimum percentage of cells in either group expressing the feature
  logfc.threshold = 0.25  # Minimum log2 fold-change for motif deviation
)

In [268]:
diff_motifs$motif<-row.names(diff_motifs)

row.names(diff_motifs)<-NULL

In [269]:

diff_motifs$comparison<-'Genotype_rs139141690_vs_Del_80bp'
diff_motifs$identity<-'3'

str(diff_motifs)
cat("\n")

results_cluster_3<-rbind(diff_motifs,results_cluster_3)


'data.frame':	429 obs. of  8 variables:
 $ p_val     : num  2.04e-15 7.84e-15 3.02e-14 5.16e-13 1.62e-12 ...
 $ avg_log2FC: num  -6.98 -8.77 -6.88 -8.62 -9.02 ...
 $ pct.1     : num  0.612 0.61 0.612 0.636 0.644 0.545 0.301 0.711 0.61 0.533 ...
 $ pct.2     : num  0.76 0.786 0.773 0.797 0.799 0.704 0.478 0.863 0.723 0.383 ...
 $ p_val_adj : num  1.52e-12 5.85e-12 2.25e-11 3.85e-10 1.21e-09 ...
 $ motif     : chr  "GATA6" "GATA2" "GATA3" "GATA5" ...
 $ comparison: chr  "Genotype_rs139141690_vs_Del_80bp" "Genotype_rs139141690_vs_Del_80bp" "Genotype_rs139141690_vs_Del_80bp" "Genotype_rs139141690_vs_Del_80bp" ...
 $ identity  : chr  "3" "3" "3" "3" ...



In [270]:
indexes<-grep("CUX1|RUNX1",diff_motifs$motif)
diff_motifs[indexes,]

,p_val,avg_log2FC,pct.1,pct.2,p_val_adj,motif,comparison,identity
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>
72,0.001614781,2.6179631,0.783,0.744,1,RUNX1,Genotype_rs139141690_vs_Del_80bp,3
178,0.042715829,-0.2987351,0.640,0.697,1,CUX1,Genotype_rs139141690_vs_Del_80bp,3


### Evaluate

In [271]:
indexes<-grep("CUX1|RUNX1",results_cluster_3$motif)
results_cluster_3[indexes,]



,p_val,avg_log2FC,pct.1,pct.2,p_val_adj,motif,comparison,identity
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>
72,1.614781e-03,2.6179631,0.783,0.744,1.000000e+00,RUNX1,Genotype_rs139141690_vs_Del_80bp,3
178,4.271583e-02,-0.2987351,0.640,0.697,1.000000e+00,CUX1,Genotype_rs139141690_vs_Del_80bp,3
651,3.756347e-01,0.4132764,0.783,0.757,1.000000e+00,RUNX1,Genotype_Del_16bp_vs_Del_80bp,3
903,7.008065e-07,-0.4635739,0.588,0.744,5.228016e-04,RUNX1,Genotype_rs139141690_vs_wt,3
1023,1.498794e-02,-0.5547957,0.630,0.697,1.000000e+00,CUX1,Genotype_rs139141690_vs_wt,3
1254,2.184512e-16,-3.0815370,0.588,0.783,1.629646e-13,RUNX1,Genotype_Del_80bp_vs_wt,3
1645,7.259871e-01,-0.2560605,0.630,0.640,1.000000e+00,CUX1,Genotype_Del_80bp_vs_wt,3
1735,1.244185e-06,-2.6682606,0.588,0.757,9.281622e-04,RUNX1,Genotype_Del_16bp_vs_wt,3


## Subset to cluster 1 and 3

In [191]:
adata_sub_cluster_1_and_3<-subset(adata, seurat_clusters == '1' | seurat_clusters == '3')

adata_sub_cluster_1_and_3

Idents(adata_sub_cluster_1_and_3)<- "Genotype"


summary(adata_sub_cluster_1_and_3@meta.data$Genotype)


An object of class Seurat 
459843 features across 4395 samples within 5 assays 
Active assay: chromvar (746 features, 0 variable features)
 1 layer present: data
 4 other assays present: RNA, RNA_raw, ATAC, SCT
 5 dimensional reductions calculated: pca, umap.rna, lsi, umap.atac, umap.wnn

wt rs139141690_HET     rs139141690        Del_16bp        Del_80bp 
           1288             626             775             283            1423

In [192]:
results_cluster_1_and_3<-data.frame()

### wt vs Del_16bp

In [193]:
# Set the chromvar assay as the default for this analysis
DefaultAssay(adata_sub_cluster_1_and_3) <- "chromvar"

# Find differentially active motifs between two cell types (e.g., "T cells" vs "B cells")
# This is analogous to FindMarkers for gene expression, but now on motif deviation scores
diff_motifs <- FindMarkers(
  object = adata_sub_cluster_1_and_3,
  ident.1 = "wt",
  ident.2 = "Del_16bp",
  group.by = "Genotype", # Or whatever metadata column defines your groups
  test.use = "t",    # Or "bimod", "t", etc.
  min.pct = 0.1,          # Minimum percentage of cells in either group expressing the feature
  logfc.threshold = 0.25  # Minimum log2 fold-change for motif deviation
)

In [194]:
diff_motifs$motif<-row.names(diff_motifs)

row.names(diff_motifs)<-NULL

In [195]:

diff_motifs$comparison<-'Genotype_Del_16bp_vs_wt'
diff_motifs$identity<-'MIX'

str(diff_motifs)
cat("\n")

results_cluster_1_and_3<-rbind(diff_motifs,results_cluster_1_and_3)


'data.frame':	409 obs. of  8 variables:
 $ p_val     : num  5.57e-12 6.28e-12 1.82e-11 2.34e-11 2.37e-11 ...
 $ avg_log2FC: num  -3.144 -1.838 -0.652 -1.473 0.291 ...
 $ pct.1     : num  0.384 0.378 0.301 0.363 0.299 0.447 0.393 0.387 0.292 0.395 ...
 $ pct.2     : num  0.583 0.572 0.523 0.551 0.551 0.657 0.58 0.576 0.541 0.59 ...
 $ p_val_adj : num  4.15e-09 4.69e-09 1.36e-08 1.74e-08 1.77e-08 ...
 $ motif     : chr  "ETV5" "ERF" "ELF1" "ETV6" ...
 $ comparison: chr  "Genotype_Del_16bp_vs_wt" "Genotype_Del_16bp_vs_wt" "Genotype_Del_16bp_vs_wt" "Genotype_Del_16bp_vs_wt" ...
 $ identity  : chr  "MIX" "MIX" "MIX" "MIX" ...



In [196]:
indexes<-grep("CUX1|RUNX1",diff_motifs$motif)
diff_motifs[indexes,]

,p_val,avg_log2FC,pct.1,pct.2,p_val_adj,motif,comparison,identity
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>
32,4.052306e-07,1.115537,0.444,0.572,0.0003023021,RUNX1,Genotype_Del_16bp_vs_wt,MIX


### wt vs Del_80bp

In [197]:
# Set the chromvar assay as the default for this analysis
DefaultAssay(adata_sub_cluster_1_and_3) <- "chromvar"

# Find differentially active motifs between two cell types (e.g., "T cells" vs "B cells")
# This is analogous to FindMarkers for gene expression, but now on motif deviation scores
diff_motifs <- FindMarkers(
  object = adata_sub_cluster_1_and_3,
  ident.1 = "wt",
  ident.2 = "Del_80bp",
  group.by = "Genotype", # Or whatever metadata column defines your groups
  test.use = "t",    # Or "bimod", "t", etc.
  min.pct = 0.1,          # Minimum percentage of cells in either group expressing the feature
  logfc.threshold = 0.25  # Minimum log2 fold-change for motif deviation
)

In [198]:
diff_motifs$motif<-row.names(diff_motifs)

row.names(diff_motifs)<-NULL

In [199]:

diff_motifs$comparison<-'Genotype_Del_80bp_vs_wt'
diff_motifs$identity<-'MIX'

str(diff_motifs)
cat("\n")

results_cluster_1_and_3<-rbind(diff_motifs,results_cluster_1_and_3)


'data.frame':	359 obs. of  8 variables:
 $ p_val     : num  8.94e-51 1.19e-49 2.04e-48 1.07e-46 5.81e-45 ...
 $ avg_log2FC: num  -3.02 -2.24 -2.14 -2.88 -2.33 ...
 $ pct.1     : num  0.378 0.299 0.384 0.363 0.301 0.387 0.395 0.392 0.385 0.393 ...
 $ pct.2     : num  0.606 0.572 0.591 0.586 0.55 0.602 0.592 0.588 0.596 0.594 ...
 $ p_val_adj : num  6.67e-48 8.89e-47 1.52e-45 8.01e-44 4.33e-42 ...
 $ motif     : chr  "ERF" "GABPA" "ETV5" "ETV6" ...
 $ comparison: chr  "Genotype_Del_80bp_vs_wt" "Genotype_Del_80bp_vs_wt" "Genotype_Del_80bp_vs_wt" "Genotype_Del_80bp_vs_wt" ...
 $ identity  : chr  "MIX" "MIX" "MIX" "MIX" ...



In [200]:
indexes<-grep("CUX1|RUNX1",diff_motifs$motif)
diff_motifs[indexes,]

,p_val,avg_log2FC,pct.1,pct.2,p_val_adj,motif,comparison,identity
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>
16,1.514748e-30,-0.4089126,0.444,0.617,1.130002e-27,RUNX1,Genotype_Del_80bp_vs_wt,MIX


### wt vs rs139141690_HET

In [201]:
# Set the chromvar assay as the default for this analysis
DefaultAssay(adata_sub_cluster_1_and_3) <- "chromvar"

# Find differentially active motifs between two cell types (e.g., "T cells" vs "B cells")
# This is analogous to FindMarkers for gene expression, but now on motif deviation scores
diff_motifs <- FindMarkers(
  object = adata_sub_cluster_1_and_3,
  ident.1 = "wt",
  ident.2 = "rs139141690_HET",
  group.by = "Genotype", # Or whatever metadata column defines your groups
  test.use = "t",    # Or "bimod", "t", etc.
  min.pct = 0.1,          # Minimum percentage of cells in either group expressing the feature
  logfc.threshold = 0.25  # Minimum log2 fold-change for motif deviation
)

In [202]:
diff_motifs$motif<-row.names(diff_motifs)

row.names(diff_motifs)<-NULL

In [203]:

diff_motifs$comparison<-'Genotype_rs139141690_HET_vs_wt'
diff_motifs$identity<-'MIX'

str(diff_motifs)
cat("\n")

results_cluster_1_and_3<-rbind(diff_motifs,results_cluster_1_and_3)


'data.frame':	367 obs. of  8 variables:
 $ p_val     : num  1.90e-24 2.00e-16 4.65e-16 1.29e-15 3.80e-14 ...
 $ avg_log2FC: num  1.973 -0.683 1.672 3.7 0.946 ...
 $ pct.1     : num  0.764 0.519 0.692 0.686 0.573 0.683 0.557 0.578 0.479 0.515 ...
 $ pct.2     : num  0.559 0.687 0.57 0.556 0.411 0.554 0.42 0.423 0.625 0.631 ...
 $ p_val_adj : num  1.41e-21 1.50e-13 3.47e-13 9.64e-13 2.84e-11 ...
 $ motif     : chr  "GATA1::TAL1" "TFDP1" "GATA4" "GATA2" ...
 $ comparison: chr  "Genotype_rs139141690_HET_vs_wt" "Genotype_rs139141690_HET_vs_wt" "Genotype_rs139141690_HET_vs_wt" "Genotype_rs139141690_HET_vs_wt" ...
 $ identity  : chr  "MIX" "MIX" "MIX" "MIX" ...



In [204]:
indexes<-grep("CUX1|RUNX1",diff_motifs$motif)
diff_motifs[indexes,]

,p_val,avg_log2FC,pct.1,pct.2,p_val_adj,motif,comparison,identity
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>
276,0.07541385,5.694025,0.444,0.414,1,RUNX1,Genotype_rs139141690_HET_vs_wt,MIX


### wt vs rs139141690

In [205]:
# Set the chromvar assay as the default for this analysis
DefaultAssay(adata_sub_cluster_1_and_3) <- "chromvar"

# Find differentially active motifs between two cell types (e.g., "T cells" vs "B cells")
# This is analogous to FindMarkers for gene expression, but now on motif deviation scores
diff_motifs <- FindMarkers(
  object = adata_sub_cluster_1_and_3,
  ident.1 = "wt",
  ident.2 = "rs139141690",
  group.by = "Genotype", # Or whatever metadata column defines your groups
  test.use = "t",    # Or "bimod", "t", etc.
  min.pct = 0.1,          # Minimum percentage of cells in either group expressing the feature
  logfc.threshold = 0.25  # Minimum log2 fold-change for motif deviation
)

In [206]:
diff_motifs$motif<-row.names(diff_motifs)

row.names(diff_motifs)<-NULL

In [207]:

diff_motifs$comparison<-'Genotype_rs139141690_vs_wt'
diff_motifs$identity<-'MIX'

str(diff_motifs)
cat("\n")

results_cluster_1_and_3<-rbind(diff_motifs,results_cluster_1_and_3)


'data.frame':	328 obs. of  8 variables:
 $ p_val     : num  4.37e-42 1.57e-36 4.07e-36 1.87e-35 1.33e-34 ...
 $ avg_log2FC: num  -2.88 -1.78 -2.11 -3.34 -2.17 ...
 $ pct.1     : num  0.363 0.384 0.334 0.411 0.396 0.48 0.365 0.387 0.419 0.378 ...
 $ pct.2     : num  0.592 0.619 0.585 0.634 0.636 0.702 0.59 0.626 0.652 0.581 ...
 $ p_val_adj : num  3.26e-39 1.17e-33 3.03e-33 1.40e-32 9.92e-32 ...
 $ motif     : chr  "ETV6" "ETV5" "EHF" "IKZF1" ...
 $ comparison: chr  "Genotype_rs139141690_vs_wt" "Genotype_rs139141690_vs_wt" "Genotype_rs139141690_vs_wt" "Genotype_rs139141690_vs_wt" ...
 $ identity  : chr  "MIX" "MIX" "MIX" "MIX" ...



In [208]:
indexes<-grep("CUX1|RUNX1",diff_motifs$motif)
diff_motifs[indexes,]

,p_val,avg_log2FC,pct.1,pct.2,p_val_adj,motif,comparison,identity
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>
37,7.013810e-19,-2.8661708,0.444,0.635,5.232302e-16,RUNX1,Genotype_rs139141690_vs_wt,MIX
120,7.452915e-07,-0.4520451,0.570,0.657,5.559874e-04,CUX1,Genotype_rs139141690_vs_wt,MIX


### Del_80bp vs Del_16bp

In [209]:
# Set the chromvar assay as the default for this analysis
DefaultAssay(adata_sub_cluster_1_and_3) <- "chromvar"

# Find differentially active motifs between two cell types (e.g., "T cells" vs "B cells")
# This is analogous to FindMarkers for gene expression, but now on motif deviation scores
diff_motifs <- FindMarkers(
  object = adata_sub_cluster_1_and_3,
  ident.1 = "Del_80bp",
  ident.2 = "Del_16bp",
  group.by = "Genotype", # Or whatever metadata column defines your groups
  test.use = "t",    # Or "bimod", "t", etc.
  min.pct = 0.1,          # Minimum percentage of cells in either group expressing the feature
  logfc.threshold = 0.25  # Minimum log2 fold-change for motif deviation
)

In [210]:
diff_motifs$motif<-row.names(diff_motifs)

row.names(diff_motifs)<-NULL

In [211]:

diff_motifs$comparison<-'Genotype_Del_16bp_vs_Del_80bp'
diff_motifs$identity<-'MIX'

str(diff_motifs)
cat("\n")

results_cluster_1_and_3<-rbind(diff_motifs,results_cluster_1_and_3)


'data.frame':	346 obs. of  8 variables:
 $ p_val     : num  0.00155 0.00285 0.00299 0.00438 0.00472 ...
 $ avg_log2FC: num  -0.382 -0.267 0.658 2.79 0.428 ...
 $ pct.1     : num  0.172 0.484 0.502 0.491 0.533 0.236 0.512 0.416 0.523 0.184 ...
 $ pct.2     : num  0.254 0.572 0.59 0.622 0.47 0.339 0.58 0.505 0.576 0.269 ...
 $ p_val_adj : num  1 1 1 1 1 1 1 1 1 1 ...
 $ motif     : chr  "GFI1" "ZNF136" "NFIC(var.2)" "BCL6B" ...
 $ comparison: chr  "Genotype_Del_16bp_vs_Del_80bp" "Genotype_Del_16bp_vs_Del_80bp" "Genotype_Del_16bp_vs_Del_80bp" "Genotype_Del_16bp_vs_Del_80bp" ...
 $ identity  : chr  "MIX" "MIX" "MIX" "MIX" ...



In [212]:
indexes<-grep("CUX1|RUNX1",diff_motifs$motif)
diff_motifs[indexes,]

,p_val,avg_log2FC,pct.1,pct.2,p_val_adj,motif,comparison,identity
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>
254,0.5573459,1.52445,0.617,0.572,1,RUNX1,Genotype_Del_16bp_vs_Del_80bp,MIX


### Del_80bp vs rs139141690

In [213]:
# Set the chromvar assay as the default for this analysis
DefaultAssay(adata_sub_cluster_1_and_3) <- "chromvar"

# Find differentially active motifs between two cell types (e.g., "T cells" vs "B cells")
# This is analogous to FindMarkers for gene expression, but now on motif deviation scores
diff_motifs <- FindMarkers(
  object = adata_sub_cluster_1_and_3,
  ident.1 = "Del_80bp",
  ident.2 = "rs139141690",
  group.by = "Genotype", # Or whatever metadata column defines your groups
  test.use = "t",    # Or "bimod", "t", etc.
  min.pct = 0.1,          # Minimum percentage of cells in either group expressing the feature
  logfc.threshold = 0.25  # Minimum log2 fold-change for motif deviation
)

In [214]:
diff_motifs$motif<-row.names(diff_motifs)

row.names(diff_motifs)<-NULL

In [215]:

diff_motifs$comparison<-'Genotype_rs139141690_vs_Del_80bp'
diff_motifs$identity<-'MIX'

str(diff_motifs)
cat("\n")

results_cluster_1_and_3<-rbind(diff_motifs,results_cluster_1_and_3)


'data.frame':	350 obs. of  8 variables:
 $ p_val     : num  2.52e-40 9.93e-36 3.69e-35 4.00e-33 1.83e-31 ...
 $ avg_log2FC: num  2.34 3.66 -1.35 -2.83 -1.34 ...
 $ pct.1     : num  0.54 0.554 0.569 0.59 0.507 0.584 0.546 0.656 0.536 0.564 ...
 $ pct.2     : num  0.764 0.773 0.783 0.785 0.708 0.782 0.69 0.804 0.665 0.702 ...
 $ p_val_adj : num  1.88e-37 7.41e-33 2.76e-32 2.98e-30 1.36e-28 ...
 $ motif     : chr  "GATA6" "GATA3" "GATA2" "GATA4" ...
 $ comparison: chr  "Genotype_rs139141690_vs_Del_80bp" "Genotype_rs139141690_vs_Del_80bp" "Genotype_rs139141690_vs_Del_80bp" "Genotype_rs139141690_vs_Del_80bp" ...
 $ identity  : chr  "MIX" "MIX" "MIX" "MIX" ...



In [216]:
indexes<-grep("CUX1|RUNX1",diff_motifs$motif)
diff_motifs[indexes,]

,p_val,avg_log2FC,pct.1,pct.2,p_val_adj,motif,comparison,identity
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>
73,2.059638e-08,-0.3867458,0.565,0.657,1.53649e-05,CUX1,Genotype_rs139141690_vs_Del_80bp,MIX
305,4.297443e-01,-2.4572582,0.617,0.635,1.00000e+00,RUNX1,Genotype_rs139141690_vs_Del_80bp,MIX


### Evaluate

In [217]:
indexes<-grep("CUX1|RUNX1",results_cluster_1_and_3$motif)
results_cluster_1_and_3[indexes,]



,p_val,avg_log2FC,pct.1,pct.2,p_val_adj,motif,comparison,identity
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>
73,2.059638e-08,-0.3867458,0.565,0.657,1.536490e-05,CUX1,Genotype_rs139141690_vs_Del_80bp,MIX
305,4.297443e-01,-2.4572582,0.617,0.635,1.000000e+00,RUNX1,Genotype_rs139141690_vs_Del_80bp,MIX
604,5.573459e-01,1.5244497,0.617,0.572,1.000000e+00,RUNX1,Genotype_Del_16bp_vs_Del_80bp,MIX
733,7.013810e-19,-2.8661708,0.444,0.635,5.232302e-16,RUNX1,Genotype_rs139141690_vs_wt,MIX
816,7.452915e-07,-0.4520451,0.570,0.657,5.559874e-04,CUX1,Genotype_rs139141690_vs_wt,MIX
1300,7.541385e-02,5.6940246,0.444,0.414,1.000000e+00,RUNX1,Genotype_rs139141690_HET_vs_wt,MIX
1407,1.514748e-30,-0.4089126,0.444,0.617,1.130002e-27,RUNX1,Genotype_Del_80bp_vs_wt,MIX
1782,4.052306e-07,1.1155371,0.444,0.572,3.023021e-04,RUNX1,Genotype_Del_16bp_vs_wt,MIX


# Order factors and save

In [220]:
results_cluster_1_and_3$comparison<-factor(results_cluster_1_and_3$comparison,
                                          levels=c('Genotype_rs139141690_HET_vs_wt',
                                                   'Genotype_rs139141690_vs_wt',
                                                   'Genotype_Del_16bp_vs_wt',
                                                  'Genotype_Del_80bp_vs_wt',
                                                  'Genotype_rs139141690_vs_Del_80bp',
                                                  'Genotype_Del_16bp_vs_Del_80bp'),
                                          ordered=T)


summary(results_cluster_1_and_3$comparison)

Genotype_rs139141690_HET_vs_wt       Genotype_rs139141690_vs_wt 
                             367                              328 
         Genotype_Del_16bp_vs_wt          Genotype_Del_80bp_vs_wt 
                             409                              359 
Genotype_rs139141690_vs_Del_80bp    Genotype_Del_16bp_vs_Del_80bp 
                             350                              346

# Save

In [222]:
setwd("/group/soranzo/manuel.tardaguila/2025_K562_multiome_reanalysis/Downstream_analysis/chromvar_analysis/")

In [223]:
saveRDS(results_cluster_1_and_3, file="Differential_chromvar_scores.rds")